# FinQA Chatbot — Colab Runner

End-to-end runbook for the FinQA agentic-RAG system on Google Colab (T4 GPU).

**What this notebook does**, in order:

1. **Setup** — mount Drive, install Python dependencies.
2. **vLLM server** — launch Qwen2.5-7B-Instruct-AWQ as a background process, wait until ready.
3. **Index build** (one-time) — tokenize docs, build FAISS dense index + BM25 sparse index.
4. **Retrieval evaluation** — measure `recall@k` on dev to diagnose the retriever alone.
5. **End-to-end evaluation** — ablation across 4 cells: `{LangGraph, Baseline} × {Oracle, Retrieval}`.
6. **Gradio UI** — interactive demo with 3 tabs (Chat / Monitoring / Canary-Drift).
7. **Canary** — append a run to the history so the Drift tab shows real data.

**Prerequisites**: a Colab runtime with a T4 (or better) GPU and a Google Drive containing the repo at `/content/drive/MyDrive/finqa-chatbot/` with `data/raw/{train,dev,test}.json`.

> Each step is idempotent — you can re-run individual cells without breaking earlier state.

## 1. Setup — mount Drive, install dependencies

We keep the repository on Google Drive so it survives Colab runtime restarts. `requirements.txt` pulls in LangChain, LangGraph, FAISS, sentence-transformers, rank-bm25, and Gradio. We also install `vllm` explicitly since it is commented out in `requirements.txt` (vLLM is only needed on a GPU runtime).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/finqa-chatbot

# Core pipeline deps
!pip install -q -r requirements.txt

# vLLM (heavy install — ~5-10 min first time; skipped if already present)
!pip install -q vllm

# Dataset lib used by the retriever fine-tune script
!pip install -q datasets

## 2. Start the vLLM server

vLLM serves the LLM via an OpenAI-compatible HTTP API at `localhost:8000`. Our agent points LangChain's `ChatOpenAI` at this URL, so any model that vLLM can host is a drop-in replacement.

We use **AWQ-quantized Qwen2.5-7B-Instruct** because it fits a T4 (16 GB) at `max-model-len=8192` with room to spare for the KV-cache. The 8192-token context is chosen so that `system prompt + 5 few-shot + full FinQA page` never overflows (the default 4096 overflows ~5% of dev examples).

Startup takes 2–5 minutes on first run (model download) and ~1 minute afterwards (cached weights).

In [ ]:
import subprocess

# Kill any zombie vLLM from a previous run on the same runtime.
!pkill -9 -f api_server 2>/dev/null
!sleep 3

log_f = open('/content/vllm.log', 'w')
proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen2.5-7B-Instruct-AWQ',
    '--port', '8000',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.9',
    '--quantization', 'awq',
], stdout=log_f, stderr=subprocess.STDOUT)
print('vLLM PID:', proc.pid)

### Wait until the server is ready

Poll the `/v1/models` endpoint until it returns 200. Break early on success. If it times out, the cell prints the last 40 lines of `vllm.log` so you can diagnose (common causes: OOM, wrong model ID, port already in use).

In [ ]:
import requests, time

for i in range(60):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print('✅ vLLM ready:', [m['id'] for m in r.json()['data']])
            break
    except Exception:
        pass
    time.sleep(10)
    print(f'waiting... {i * 10}s')
else:
    print('❌ timeout — last 40 log lines:')
    !tail -40 /content/vllm.log

## 3. Build the search index (one-time per config change)

This reads `data/raw/{train,dev}.json`, chunks each FinQADocument (paragraph-level for pre/post text + per-row for tables, plus a whole-table chunk for retrieval signal), embeds chunks with `BAAI/bge-base-en-v1.5`, and persists a FAISS IndexFlatIP and a `rank-bm25` sparse index to `data/faiss_index/` and `data/processed/`.

**Skip this cell if the index files already exist.** Takes ~5 minutes on T4 for ~110k chunks (7k docs × ~17 chunks/doc).

In [ ]:
import os
idx = 'data/faiss_index/index.faiss'
chunks = 'data/processed/chunks.pkl'
bm25 = 'data/processed/bm25.pkl'
if not all(os.path.exists(p) for p in [idx, chunks, bm25]):
    !python -m src.indexing.build_index
else:
    print('Index files already present — skipping build.')
    !ls -lh {idx} {chunks} {bm25}

## 4. Retrieval diagnostic — recall@k on dev (200 examples)

This isolates the *retrieval* quality (which doc we return) from the *reasoning* quality (given the right doc, can we answer correctly). It evaluates four stages of the pipeline in one pass:

- `dense` — BGE-base FAISS only
- `bm25` — sparse only
- `+RRF` — Reciprocal Rank Fusion of the two
- `+rerank` — cross-encoder (`bge-reranker-v2-m3`) rerank over top-50 fused candidates
- `+count` — an alternative aggregation that votes by chunk-count per parent doc (experimental)

**Interpretation**: the final column's `@3` is the ceiling for retrieval-mode execution accuracy. If it is 38 %, retrieval-mode end-to-end cannot exceed ~38 % no matter how good the reasoning.

In [ ]:
!python scripts/eval_retrieval.py --split dev --max-examples 200 --k 1,3,5,10

## 5. End-to-end evaluation — 4-cell ablation

Two axes, two values each, four runs total:

| | Baseline (vanilla RAG, 1 LLM call) | LangGraph + Tool (production) |
|---|---|---|
| **Oracle** (gold doc supplied → isolates reasoning) | `--baseline --oracle` | `--oracle --no-verify` |
| **Retrieval** (end-to-end) | `--baseline` | `--no-verify` |

Flags:
- `--baseline` swaps out the LangGraph agent for a single-shot RAG call.
- `--oracle` skips retrieval and injects the gold document directly (FinQA paper's eval protocol).
- `--no-verify` disables the VERIFY critic node, which we measured to hurt accuracy by ~22 pp.

The full 883-example dev takes ~1.5 hours per cell. For a presentation, `--max-examples 100` is plenty (5-8 min per cell).

In [ ]:
# Cell 1 — Baseline, Oracle (matches FinQA paper protocol; reasoning ceiling)
!python scripts/run_eval.py --split dev --max-examples 100 --baseline --oracle \
    --output results/cell_baseline_oracle.json

In [ ]:
# Cell 2 — LangGraph + Tool, Oracle (production config, reasoning isolated)
!python scripts/run_eval.py --split dev --max-examples 100 --oracle --no-verify \
    --output results/cell_langgraph_oracle.json

In [ ]:
# Cell 3 — Baseline, Retrieval (end-to-end, no tool)
!python scripts/run_eval.py --split dev --max-examples 100 --baseline \
    --output results/cell_baseline_retrieval.json

In [ ]:
# Cell 4 — LangGraph + Tool, Retrieval (end-to-end, production)
!python scripts/run_eval.py --split dev --max-examples 100 --no-verify \
    --output results/cell_langgraph_retrieval.json

### Summarize the 4 cells

Reads the four result JSONs and prints a side-by-side table. The headline finding reproduced in REPORT.md: the LangGraph agent trades ~3 pp of execution accuracy for **+15 pp of program accuracy** on oracle — every answer comes with an auditable DSL trace at near-zero cost.

In [ ]:
import json, os

cells = {
    ('Baseline', 'Oracle'):    'results/cell_baseline_oracle.json',
    ('LangGraph', 'Oracle'):   'results/cell_langgraph_oracle.json',
    ('Baseline', 'Retrieval'): 'results/cell_baseline_retrieval.json',
    ('LangGraph', 'Retrieval'):'results/cell_langgraph_retrieval.json',
}
print(f"{'System':<12}{'Mode':<12}{'Exec':>8}{'Program':>10}{'Errors':>8}")
print('-' * 50)
for (system, mode), path in cells.items():
    if not os.path.exists(path):
        print(f'{system:<12}{mode:<12}     ? (missing)')
        continue
    with open(path) as f:
        r = json.load(f)
    print(
        f"{system:<12}{mode:<12}"
        f"{r['execution_accuracy']*100:>7.2f}%"
        f"{r['program_accuracy']*100:>9.2f}%"
        f"{len(r.get('errors', [])):>8d}"
    )

## 6. Launch the Gradio UI

Three tabs:
- **💬 Chat** — curated demos + full 883 dev dropdown + oracle toggle + reasoning trace + 👍/👎 feedback.
- **📊 Monitoring** — run history, node statistics, user feedback log.
- **📈 Canary / Drift** — canary accuracy history, regression alerts.

The server binds to `0.0.0.0:7860` and `share=True` auto-generates a public `gradio.live` URL that stays valid for 1 week.

**Leave this cell running** during the demo — stopping it kills the UI. Open the `gradio.live` URL in a new tab.

In [ ]:
!python -m app.main

## 7. Canary run — populate the Drift tab

Runs a small (50-question) oracle-mode evaluation and appends one row to `results/canary_history.jsonl`. The Gradio Canary/Drift tab reads this file and computes regression alerts (execution accuracy drop ≥ 3 pp → critical; parse success drop ≥ 5 pp or p95 latency growth ≥ 1.5× → warning).

**Run this in a separate cell** while the Gradio UI from Step 6 is still running in its own cell (Colab supports multiple concurrent cells). The Drift tab's 🔄 Refresh button will then show the new entry.

The `--notes` flag is free-form text that appears in the history table; use it to label what changed between canary runs.

In [ ]:
# First canary = baseline reference point
!python scripts/run_canary.py --n 50 \
    --notes 'baseline: Qwen-7B AWQ + LangGraph + no VERIFY'

In [ ]:
# Second canary on the same config — demonstrates no-drift case
!python scripts/run_canary.py --n 30 \
    --notes 'sanity check: same config, expect no drift'

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `openai.APIConnectionError: Connection error` | vLLM not running on :8000 | Re-run Step 2 (vLLM startup + wait) |
| `Connection refused` in canary | Same | Same |
| `maximum context length is 4096 tokens` | vLLM started with too-short context | Include `--max-model-len 8192` on the vLLM command |
| `Index files missing` | Step 3 skipped or index deleted | Re-run Step 3 |
| Gradio shows "Error" on every component | Old `gradio_ui.py` cached in memory | Runtime → Restart runtime, then re-run cells |
| vLLM OOM on T4 | Model too big | Already using AWQ-quantized 7B — if still OOM, drop to `Qwen/Qwen2.5-3B-Instruct` |
| `dict has no attribute get_block_name` in Gradio | `gr.State` ↔ Gradio 5.x bug | Resolved in the current `gradio_ui.py` (stateless design) |
| Canary entry with 0% accuracy | vLLM request failed silently | Check `/content/vllm.log`, verify Step 2 |

## Expected headline numbers (for sanity-checking your run)

Qwen2.5-7B-Instruct-AWQ, zero-shot, full FinQA dev (n=883), oracle mode:

| System | Execution | Program | Notes |
|---|---|---|---|
| Baseline (vanilla RAG, inline program) | **~62 %** | ~24 % | Matches FinQA paper's supervised RoBERTa-large (61.24 %) |
| LangGraph + Tool (production) | **~59 %** | **~38 %** | Auditable DSL trace for every answer |

If your numbers are more than ~5 pp below these, something is off — the most common cause is silent vLLM failures producing empty answers. Check `results/*.json` and `/content/vllm.log` for error rates.